# GeoNUSAF — Model Card Compendium

Collects the Hugging Face **model card** of every trained GeoNUSAF checkpoint
(Part 1 segmentation + Part 2 generation) and compiles them into **one** Word
document, with a clear title for each model.

**Outputs written to `/kaggle/working/`**

| File | Purpose |
|---|---|
| `GeoNUSAF_All_Model_Cards.docx` | the deliverable — TOC, index table, one titled section per model |
| `GeoNUSAF_All_Model_Cards.md` | same content as flat markdown (easiest thing to hand to an LLM for the Results section) |
| `model_cards_raw/*.md` | one raw card per repo, cached so re-runs are free |

**Before running**

1. Notebook settings → **Internet: On** (the Hub is fetched over HTTPS).
2. Only if some repo is private: Add-ons → Secrets → add `HF_TOKEN`.
3. Accelerator: **None**. This notebook is pure I/O, it needs no GPU.

Runtime is about 1–2 minutes for 50 repositories.

---

## 1. Environment setup

`python-docx` is the only dependency that is not already on the Kaggle image.


In [ ]:
!pip install -q python-docx==1.1.2

print("install step done")


## 2. Imports

`requests` handles the Hub, `python-docx` writes the document, everything else
is standard library. No torch, no CUDA — this notebook touches neither.


In [ ]:
import os
import re
import sys
import time
import json
import shutil
import textwrap
import platform
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime

import requests

import docx
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

print(f"python      : {platform.python_version()}")
print(f"requests    : {requests.__version__}")
print(f"python-docx : {docx.__version__ if hasattr(docx, '__version__') else 'installed'}")


## 3. Configuration

Single `@dataclass CFG` as in every other GeoNUSAF notebook. Nothing below this
cell hard-codes a path, a filename or a threshold.


In [ ]:
@dataclass
class CFG:
    # ---- output ----------------------------------------------------------
    OUT_DIR: str = "/kaggle/working"
    DOCX_NAME: str = "GeoNUSAF_All_Model_Cards.docx"
    MD_NAME: str = "GeoNUSAF_All_Model_Cards.md"
    CACHE_DIRNAME: str = "model_cards_raw"

    # ---- hub -------------------------------------------------------------
    HF_SECRET_NAME: str = "HF_TOKEN"      # Kaggle secret, optional
    REVISIONS: tuple = ("main", "master")
    TIMEOUT_S: int = 30
    RETRIES: int = 3
    SLEEP_BETWEEN_S: float = 0.35
    USE_CACHE: bool = True                # skip re-download if cached
    FETCH_META: bool = True               # last-modified / commit / file list

    # ---- document --------------------------------------------------------
    DOC_TITLE: str = "GeoNUSAF - Complete Model Card Compendium"
    DOC_SUBTITLE: str = ("Segmentation (Part 1) and diffusion-based generation "
                         "(Part 2): all Hugging Face checkpoints")
    BASE_FONT: str = "Calibri"
    BASE_PT: float = 10.5
    MONO_FONT: str = "Consolas"
    MONO_PT: float = 8.5
    TABLE_PT: float = 8.5
    TOC_LEVELS: str = "1-4"
    PAGE_BREAK_PER_MODEL: bool = True
    KEEP_FRONTMATTER: bool = True         # YAML block verbatim (holds metrics)
    DROP_BADGES: bool = True

    # ---- behaviour -------------------------------------------------------
    STRICT: bool = False                  # True -> raise if any card missing
    VERIFY: bool = True                   # re-open the docx and check it

cfg = CFG()

OUT = Path(cfg.OUT_DIR)
CACHE = OUT / cfg.CACHE_DIRNAME
DOCX_PATH = OUT / cfg.DOCX_NAME
MD_PATH = OUT / cfg.MD_NAME
CACHE.mkdir(parents=True, exist_ok=True)

for k, v in asdict(cfg).items():
    print(f"{k:24s} {v}")


## 4. Authentication

Public repositories need no token. If any repo under `sugam24` or
`Pranilllllll` is private, add a Kaggle secret named `HF_TOKEN` and it is picked
up here automatically; otherwise the notebook simply runs unauthenticated.


In [ ]:
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret(cfg.HF_SECRET_NAME)
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if HF_TOKEN is None:
        print(f"no secret '{cfg.HF_SECRET_NAME}' ({type(e).__name__}) - "
              f"continuing unauthenticated")

if HF_TOKEN:
    print(f"token loaded: {HF_TOKEN[:5]}...{HF_TOKEN[-3:]} "
          f"({len(HF_TOKEN)} chars)")
else:
    print("no token - public repositories only "
          "(private ones will report 401/403 and be skipped)")


## 5. Model registry

The whole notebook is driven by this one structure:

```
Part  ->  Group (architecture / stage)  ->  Variant (split / arm)  ->  [repo ids]
```

Those four levels become heading levels 1-4 in the document, which is what gives
every model a clear, unambiguous title. Add or remove a line here and every
downstream section adapts — nothing else references a repo id.


In [ ]:
MODELS = [
    ("Part 1 - Semantic Segmentation", [

        ("U-Net (ResNet-18)", [
            ("Sequence-block split", [
                "sugam24/geonusaf-unet-r18-block-fold0",
                "sugam24/geonusaf-unet-r18-block-fold1",
                "sugam24/geonusaf-unet-r18-block-fold2",
            ]),
            ("Random split", [
                "sugam24/geonusaf-unet-r18-random-fold0",
                "sugam24/geonusaf-unet-r18-random-fold1",
                "sugam24/geonusaf-unet-r18-random-fold2",
            ]),
        ]),

        ("UNetFormer (ResNet-18)", [
            ("Sequence-block split", [
                "Pranilllllll/geonusaf-unetformer-r18-block-fold0",
                "Pranilllllll/geonusaf-unetformer-r18-block-fold1",
                "Pranilllllll/geonusaf-unetformer-r18-block-fold2",
            ]),
            ("Random split", [
                "Pranilllllll/geonusaf-unetformer-r18-random-fold0",
                "Pranilllllll/geonusaf-unetformer-r18-random-fold1",
                "Pranilllllll/geonusaf-unetformer-r18-random-fold2",
            ]),
        ]),

        ("SegFormer-B0 (MiT-B0)", [
            ("Sequence-block split", [
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold0",
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1",
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold2",
            ]),
            ("Random split", [
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-random-fold0",
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-random-fold1",
                "Pranilllllll/segformer-b0-kaggle-ktm-dataset-random-fold2",
            ]),
        ]),

        ("SegNeXt-T (MSCAN-T + LightHam)", [
            ("Sequence-block split", [
                "Pranilllllll/geonusaf-segNext-block-fold0",
                "Pranilllllll/geonusaf-segNext-block-fold1",
                "Pranilllllll/geonusaf-segNext-block-fold2",
            ]),
            ("Random split", [
                "Pranilllllll/geonusaf-segNext-random-fold0",
                "Pranilllllll/geonusaf-segNext-random-fold1",
                "Pranilllllll/geonusaf-segNext-random-fold2",
            ]),
        ]),

        ("TC-SegFormer (proposed)", [
            ("Sequence-block split", [
                "sugam24/geonusaf-tcsegformer-block-fold0",
                "sugam24/geonusaf-tcsegformer-block-fold1",
                "sugam24/geonusaf-tcsegformer-block-fold2",
            ]),
            ("Random split", [
                "sugam24/geonusaf-tcsegformer-random-fold0",
                "sugam24/geonusaf-tcsegformer-random-fold1",
                "sugam24/geonusaf-tcsegformer-random-fold2",
            ]),
        ]),

        ("TC-SegFormer - component ablations (block, fold 1)", [
            ("Ablation runs", [
                "sugam24/geonusaf-tcsegformer-block-fold1-no_detail",
                "sugam24/geonusaf-tcsegformer-block-fold1-no_csa",
                "sugam24/geonusaf-tcsegformer-block-fold1-no_cldice",
                "sugam24/geonusaf-tcsegformer-block-fold1-no_sampler",
            ]),
        ]),

        ("LoveDA benchmark (external comparability)", [
            ("Both domains, v3, sliding-window eval", [
                "Pranilllllll/geonusaf-tcsegformer-loveda-both-v3-slide",
                "Pranilllllll/geonusaf-segformer-b0-loveda-both-v3-slide",
            ]),
        ]),
    ]),

    ("Part 2 - Generation Pipeline", [

        ("Stage 1 - Mask generation (label diffusion)", [
            ("Arm A - binary label code", [
                "sugam24/geonusaf-stage1-maskddpm-block-fold1",
            ]),
            ("Arm B - one-hot", [
                "sugam24/geonusaf-stage1-maskddpm-onehot-block-fold1",
            ]),
        ]),

        ("Stage 2 - Mask-to-image generation", [
            ("Ablation D (block, fold 1)", [
                "Pranilllllll/geonusaf-stage2-mask2image-block-fold1-D",
            ]),
        ]),

        ("Stage 3 - Chained sampling (synthetic mask+image pairs)", [
            ("Fake pairs R1 / R2 / R3", [
                "sugam24/geonusaf-stage3-fakepairs-block-fold1",
            ]),
        ]),

        ("Stage 4 - Downstream retraining (real + synthetic)", [
            ("Arm R0 - real only", [
                "Pranilllllll/geonusaf-s4-unet-r18-R0-block-fold1",
                "Pranilllllll/geonusaf-s4-unetformer-r18-R0-block-fold1",
                "Pranilllllll/geonusaf-s4-segformer-b0-R0-block-fold1",
                "sugam24/geonusaf-s4-segnext-t-R0-block-fold1",
                "sugam24/geonusaf-s4-tcsegformer-R0-block-fold1",
            ]),
            # Only the R0 arm was supplied. R1 is the arm the whole Stage-4
            # claim rests on, so its ids are listed here as the obvious
            # counterparts. VERIFY THEM - a wrong id reports MISSING and is
            # skipped, it will not break the run.
            ("Arm R1 - real + synthetic (verify these ids)", [
                "Pranilllllll/geonusaf-s4-unet-r18-R1-block-fold1",
                "Pranilllllll/geonusaf-s4-unetformer-r18-R1-block-fold1",
                "Pranilllllll/geonusaf-s4-segformer-b0-R1-block-fold1",
                "sugam24/geonusaf-s4-segnext-t-R1-block-fold1",
                "sugam24/geonusaf-s4-tcsegformer-R1-block-fold1",
            ]),
        ]),
    ]),
]

print(f"parts   : {len(MODELS)}")


## 6. Registry flattening and validation

Flatten to a single ordered list and check it before touching the network: a
duplicate id or a malformed `owner/name` is far cheaper to catch here than
halfway through the download loop.


In [ ]:
def iter_repos(models):
    for part, groups in models:
        for group, variants in groups:
            for variant, repos in variants:
                for repo in repos:
                    yield part, group, variant, repo

ROWS = list(iter_repos(MODELS))
REPOS = [r[3] for r in ROWS]

ID_RE = re.compile(r"^[A-Za-z0-9][\w.-]*/[\w.-]+$")

bad = [r for r in REPOS if not ID_RE.match(r)]
assert not bad, f"malformed repo ids: {bad}"

dupes = sorted({r for r in REPOS if REPOS.count(r) > 1})
assert not dupes, f"duplicate repo ids: {dupes}"

owners = {}
for r in REPOS:
    owners[r.split("/")[0]] = owners.get(r.split("/")[0], 0) + 1

print(f"repositories : {len(REPOS)}")
print(f"unique       : {len(set(REPOS))}")
print(f"owners       : {owners}")
print()
for part, groups in MODELS:
    n = sum(len(rp) for _, vs in groups for _, rp in vs)
    print(f"  {part:38s} {n:>3d} repos across {len(groups)} groups")


## 7. HTTP session

One pooled session with the auth header attached once. `raw/<rev>/README.md` is
the model card; the API endpoint gives the commit and file list that make each
section traceable back to a specific push.


In [ ]:
RAW_URL = "https://huggingface.co/{repo}/raw/{rev}/README.md"
API_URL = "https://huggingface.co/api/models/{repo}"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "geonusaf-card-compendium/1.0"})
if HF_TOKEN:
    SESSION.headers.update({"Authorization": f"Bearer {HF_TOKEN}"})

def cache_path(repo_id):
    return CACHE / (repo_id.replace("/", "__") + ".md")

# connectivity probe - fails loudly now rather than 50 times in a row later
try:
    probe = SESSION.get("https://huggingface.co/api/models/bert-base-uncased",
                        timeout=cfg.TIMEOUT_S)
    print(f"hub reachable: HTTP {probe.status_code}")
except requests.RequestException as e:
    print("HUB UNREACHABLE - switch Internet ON in the notebook settings")
    raise


## 8. Card fetch

Returns `(markdown, status)`. Every failure mode is a status string rather than
an exception, so one dead repository cannot take down a 50-repository run:

- `cache` — served from disk, no request made
- `ok` — downloaded
- `404` — repo or `README.md` absent (both revisions tried)
- `401/403` — private or gated; set `HF_TOKEN`
- `429` — rate limited, backed off and retried


In [ ]:
def fetch_card(repo_id):
    cp = cache_path(repo_id)
    if cfg.USE_CACHE and cp.exists() and cp.stat().st_size > 0:
        return cp.read_text(encoding="utf-8"), "cache"

    last_err = "unknown error"
    for rev in cfg.REVISIONS:
        url = RAW_URL.format(repo=repo_id, rev=rev)
        for attempt in range(1, cfg.RETRIES + 1):
            try:
                r = SESSION.get(url, timeout=cfg.TIMEOUT_S)
            except requests.RequestException as e:
                last_err = f"network error: {type(e).__name__}"
                time.sleep(1.5 * attempt)
                continue

            if r.status_code == 200:
                cp.write_text(r.text, encoding="utf-8")
                return r.text, "ok"
            if r.status_code == 404:
                last_err = "404 - repo or README.md not found"
                break                      # next revision; retrying is pointless
            if r.status_code in (401, 403):
                return None, "401/403 - private or gated; set HF_TOKEN"
            if r.status_code == 429:
                last_err = "429 - rate limited"
                time.sleep(5 * attempt)
                continue
            last_err = f"HTTP {r.status_code}"
            time.sleep(1.5 * attempt)

    return None, last_err


def fetch_meta(repo_id):
    if not cfg.FETCH_META:
        return {}
    try:
        r = SESSION.get(API_URL.format(repo=repo_id), timeout=cfg.TIMEOUT_S)
        if r.status_code == 200:
            j = r.json()
            files = sorted(s.get("rfilename", "") for s in j.get("siblings", []))
            return {"lastModified": j.get("lastModified", ""),
                    "downloads": j.get("downloads", ""),
                    "sha": (j.get("sha") or "")[:10],
                    "n_files": len(files),
                    "files": ", ".join(files)[:900]}
    except Exception:
        pass
    return {}

print("fetch_card / fetch_meta defined")


## 9. Download loop

Everything is pulled up front, before a single line of the document is written.
Failures are therefore all visible in one place, and the document build itself
never touches the network.


In [ ]:
CARDS, METAS, STATUS = {}, {}, {}
t0 = time.time()

for k, (_, _, _, repo) in enumerate(ROWS, 1):
    body, st = fetch_card(repo)
    CARDS[repo], STATUS[repo] = body, st
    METAS[repo] = fetch_meta(repo) if body is not None else {}
    flag = "ok  " if body is not None else "FAIL"
    size = f"{len(body):>6d} chars" if body is not None else "     -      "
    print(f"[{k:>2d}/{len(ROWS)}] {flag} {size}  {repo}  ({st})")
    if st != "cache":
        time.sleep(cfg.SLEEP_BETWEEN_S)

N_OK = sum(1 for v in CARDS.values() if v is not None)
print(f"\nfetched {N_OK}/{len(ROWS)} cards in {time.time() - t0:.1f} s")


## 10. Fetch summary

A missing card is a hole in the Results section, so it is reported explicitly
here rather than being discovered later inside the document.


In [ ]:
MISSING = [r for r in REPOS if CARDS[r] is None]

by_status = {}
for r in REPOS:
    by_status.setdefault(STATUS[r], []).append(r)

print("status counts")
for st, rs in sorted(by_status.items(), key=lambda kv: -len(kv[1])):
    print(f"  {st:48s} {len(rs):>3d}")

if MISSING:
    print(f"\n{len(MISSING)} MISSING card(s):")
    for r in MISSING:
        print(f"  {r:62s} {STATUS[r]}")
    print("\nThese appear in the index table marked MISSING and get a "
          "placeholder section in the document.")
    if cfg.STRICT:
        raise RuntimeError(f"{len(MISSING)} cards missing and CFG.STRICT is True")
else:
    print("\nevery card retrieved")

lens = [len(v) for v in CARDS.values() if v]
if lens:
    print(f"\ncard size: min {min(lens)}  median "
          f"{sorted(lens)[len(lens)//2]}  max {max(lens)} chars")


## 11. Markdown grammar

Model cards are GitHub-flavoured markdown. These patterns cover everything the
GeoNUSAF cards actually contain: YAML frontmatter, ATX headings, fenced code,
pipe tables, lists, blockquotes and rules.


In [ ]:
FRONTMATTER_RE = re.compile(r"\A---[ \t]*\r?\n(.*?)\r?\n---[ \t]*\r?\n", re.S)
HEADING_RE     = re.compile(r"^(#{1,6})\s+(.*?)\s*#*\s*$")
FENCE_RE       = re.compile(r"^\s*(```|~~~)(.*)$")
HR_RE          = re.compile(r"^\s*([-*_])(\s*\1){2,}\s*$")
UL_RE          = re.compile(r"^(\s*)[-*+]\s+(.*)$")
OL_RE          = re.compile(r"^(\s*)(\d+)[.)]\s+(.*)$")
QUOTE_RE       = re.compile(r"^\s*>\s?(.*)$")
TABLE_SEP_RE   = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)*\|?\s*$")
HTML_TAG_RE    = re.compile(r"</?[A-Za-z][^>]*>")

INLINE_RE = re.compile(
    r"(?P<code>`+[^`]+`+)"
    r"|(?P<badge>\[!\[[^\]]*\]\([^)]*\)\]\([^)]*\))"
    r"|(?P<image>!\[[^\]]*\]\([^)]*\))"
    r"|(?P<bolditalic>\*\*\*[^*]+\*\*\*)"
    r"|(?P<bold>\*\*[^*]+?\*\*|__[^_]+?__)"
    r"|(?P<italic>\*[^*\n]+?\*)"
    r"|(?P<link>\[[^\]]*\]\([^)]*\))"
)

_probe = "See **bold**, `code`, *it* and [a link](https://x.y)."
print([m.lastgroup for m in INLINE_RE.finditer(_probe)])


## 12. Low-level docx helpers

Shading and paragraph borders have no python-docx API, so they are written as
raw OOXML. `w:shd` uses `val="clear"` — `solid` renders as a black block in
Word.


In [ ]:
def shade(element, hex_fill):
    """Cell or paragraph background. val=clear, never solid."""
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), hex_fill)
    element.append(shd)


def add_hrule(doc):
    """Horizontal rule as a paragraph bottom border (not a 1-row table)."""
    p = doc.add_paragraph()
    pPr = p._p.get_or_add_pPr()
    bdr = OxmlElement("w:pBdr")
    bottom = OxmlElement("w:bottom")
    bottom.set(qn("w:val"), "single")
    bottom.set(qn("w:sz"), "6")
    bottom.set(qn("w:space"), "1")
    bottom.set(qn("w:color"), "BFBFBF")
    bdr.append(bottom)
    pPr.append(bdr)
    return p


def add_field(paragraph, instruction, placeholder=""):
    """Insert a Word field code (used for TOC and PAGE)."""
    r = paragraph.add_run()
    b = OxmlElement("w:fldChar"); b.set(qn("w:fldCharType"), "begin")
    i = OxmlElement("w:instrText"); i.set(qn("xml:space"), "preserve")
    i.text = instruction
    s = OxmlElement("w:fldChar"); s.set(qn("w:fldCharType"), "separate")
    t = OxmlElement("w:t"); t.text = placeholder
    e = OxmlElement("w:fldChar"); e.set(qn("w:fldCharType"), "end")
    for el in (b, i, s, t, e):
        r._r.append(el)
    return r

print("shade / add_hrule / add_field defined")


## 13. Inline renderer

Converts one line of markdown into styled runs. Shields.io badges are dropped
(they carry no information once the image cannot load) and images become a text
placeholder rather than a broken link — no remote asset is ever downloaded into
the document.


In [ ]:
def add_inline(paragraph, text, base_bold=False, base_italic=False):
    text = HTML_TAG_RE.sub("", text)
    pos = 0
    for m in INLINE_RE.finditer(text):
        if m.start() > pos:
            r = paragraph.add_run(text[pos:m.start()])
            r.bold, r.italic = base_bold, base_italic
        kind, raw = m.lastgroup, m.group()

        if kind == "code":
            r = paragraph.add_run(raw.strip("`"))
            r.font.name = cfg.MONO_FONT
            r.font.size = Pt(9)
            r.font.color.rgb = RGBColor(0xB0, 0x30, 0x60)
        elif kind == "badge":
            if not cfg.DROP_BADGES:
                paragraph.add_run("[badge]").italic = True
        elif kind == "image":
            alt = re.match(r"!\[([^\]]*)\]", raw).group(1)
            r = paragraph.add_run(f"[image: {alt or 'unnamed'}]")
            r.italic = True
            r.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
        elif kind == "bolditalic":
            r = paragraph.add_run(raw.strip("*")); r.bold = r.italic = True
        elif kind == "bold":
            r = paragraph.add_run(raw[2:-2]); r.bold = True; r.italic = base_italic
        elif kind == "italic":
            r = paragraph.add_run(raw[1:-1]); r.italic = True; r.bold = base_bold
        elif kind == "link":
            mm = re.match(r"\[([^\]]*)\]\(([^)]*)\)", raw)
            label, url = mm.group(1), mm.group(2)
            r = paragraph.add_run(label or url)
            r.font.color.rgb = RGBColor(0x1F, 0x4E, 0x79)
            r.underline = True
            if label and url and label.strip() != url.strip():
                r2 = paragraph.add_run(f" ({url})")
                r2.font.size = Pt(8)
                r2.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
        pos = m.end()

    if pos < len(text):
        r = paragraph.add_run(text[pos:])
        r.bold, r.italic = base_bold, base_italic

print("add_inline defined")


## 14. Block renderers — code and tables

Tables matter most here: the per-class IoU / F1 blocks in the cards are the
numbers that feed the Results section, so they must survive as real Word tables
rather than collapsing into prose.


In [ ]:
def add_code_block(doc, lines, lang=""):
    body = "\n".join(lines).rstrip()
    if not body:
        return
    if lang:
        cap = doc.add_paragraph()
        cr = cap.add_run(lang)
        cr.font.size = Pt(7.5)
        cr.font.name = cfg.MONO_FONT
        cr.font.color.rgb = RGBColor(0x80, 0x80, 0x80)
        cap.paragraph_format.space_after = Pt(0)

    for ln in body.split("\n"):
        p = doc.add_paragraph()
        pf = p.paragraph_format
        pf.space_before = Pt(0); pf.space_after = Pt(0)
        pf.left_indent = Inches(0.22)
        shade(p._p.get_or_add_pPr(), "F4F4F4")
        r = p.add_run(ln if ln.strip() else " ")
        r.font.name = cfg.MONO_FONT
        r.font.size = Pt(cfg.MONO_PT)
    doc.add_paragraph().paragraph_format.space_after = Pt(4)


def split_row(line):
    s = line.strip()
    if s.startswith("|"): s = s[1:]
    if s.endswith("|"):   s = s[:-1]
    return [c.replace(r"\|", "|").strip() for c in re.split(r"(?<!\\)\|", s)]


def add_table(doc, rows):
    if not rows:
        return
    ncols = max(len(r) for r in rows)
    rows = [r + [""] * (ncols - len(r)) for r in rows]

    t = doc.add_table(rows=len(rows), cols=ncols)
    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.LEFT
    t.autofit = True

    for i, row in enumerate(rows):
        for j, txt in enumerate(row):
            cell = t.cell(i, j)
            cell.text = ""
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(1)
            p.paragraph_format.space_after = Pt(1)
            add_inline(p, txt, base_bold=(i == 0))
            for r in p.runs:
                r.font.size = Pt(cfg.TABLE_PT)
            if i == 0:
                shade(cell._tc.get_or_add_tcPr(), "E8EEF7")
    doc.add_paragraph().paragraph_format.space_after = Pt(4)
    return t

print("add_code_block / split_row / add_table defined")


## 15. Markdown to docx renderer

Line-oriented pass over one card body. `base_level` shifts the card's own `#`
below the four registry heading levels, so a card heading can never outrank the
model title it sits under.


In [ ]:
def render_markdown(doc, md_text, base_level=6):
    lines = md_text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    i, n = 0, len(lines)

    while i < n:
        line = lines[i]

        fm = FENCE_RE.match(line)                      # fenced code
        if fm:
            fence, lang = fm.group(1), fm.group(2).strip()
            i += 1
            buf = []
            while i < n and not lines[i].strip().startswith(fence):
                buf.append(lines[i]); i += 1
            i += 1
            add_code_block(doc, buf, lang)
            continue

        if not line.strip():                            # blank
            i += 1
            continue

        hm = HEADING_RE.match(line)                     # heading
        if hm:
            lvl = min(base_level + len(hm.group(1)) - 1, 9)
            add_inline(doc.add_heading("", level=lvl), hm.group(2))
            i += 1
            continue

        if HR_RE.match(line):                           # rule
            add_hrule(doc); i += 1
            continue

        if "|" in line and i + 1 < n and TABLE_SEP_RE.match(lines[i + 1]):
            header = split_row(line)
            i += 2
            body = []
            while i < n and "|" in lines[i] and lines[i].strip():
                body.append(split_row(lines[i])); i += 1
            add_table(doc, [header] + body)
            continue

        qm = QUOTE_RE.match(line)                       # blockquote
        if qm:
            buf = []
            while i < n and QUOTE_RE.match(lines[i]):
                buf.append(QUOTE_RE.match(lines[i]).group(1)); i += 1
            p = doc.add_paragraph()
            p.paragraph_format.left_indent = Inches(0.3)
            add_inline(p, " ".join(x for x in buf if x.strip()), base_italic=True)
            continue

        um = UL_RE.match(line)                          # bullets
        if um:
            ind = len(um.group(1))
            style = "List Bullet" if ind < 2 else (
                    "List Bullet 2" if ind < 6 else "List Bullet 3")
            try:
                p = doc.add_paragraph(style=style)
            except KeyError:
                p = doc.add_paragraph()
                p.paragraph_format.left_indent = Inches(0.25 + 0.25 * (ind // 2))
            p.paragraph_format.space_after = Pt(2)
            add_inline(p, um.group(2))
            i += 1
            continue

        om = OL_RE.match(line)                          # numbered
        if om:
            ind = len(om.group(1))
            style = "List Number" if ind < 2 else "List Number 2"
            try:
                p = doc.add_paragraph(style=style)
            except KeyError:
                p = doc.add_paragraph()
                p.paragraph_format.left_indent = Inches(0.25)
                p.add_run(f"{om.group(2)}. ")
            p.paragraph_format.space_after = Pt(2)
            add_inline(p, om.group(3))
            i += 1
            continue

        buf = [line]                                    # paragraph (unwrapped)
        i += 1
        while (i < n and lines[i].strip()
               and not HEADING_RE.match(lines[i])
               and not FENCE_RE.match(lines[i])
               and not UL_RE.match(lines[i])
               and not OL_RE.match(lines[i])
               and not QUOTE_RE.match(lines[i])
               and not HR_RE.match(lines[i])
               and "|" not in lines[i]):
            buf.append(lines[i]); i += 1
        p = doc.add_paragraph()
        p.paragraph_format.space_after = Pt(6)
        add_inline(p, " ".join(x.strip() for x in buf))

print("render_markdown defined")


## 16. Document scaffolding

Base style, margins, page-number footer and the TOC field. The TOC has to be a
field rather than static text: Word populates it from the built-in Heading
styles when the document is opened.


In [ ]:
def setup_document():
    doc = Document()
    st = doc.styles["Normal"]
    st.font.name = cfg.BASE_FONT
    st.font.size = Pt(cfg.BASE_PT)
    st.paragraph_format.space_after = Pt(6)

    sec = doc.sections[0]
    sec.left_margin = sec.right_margin = Inches(0.85)
    sec.top_margin = sec.bottom_margin = Inches(0.8)

    fp = sec.footer.paragraphs[0]
    fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
    add_field(fp, " PAGE ")
    return doc


def add_toc(doc):
    p = doc.add_paragraph()
    add_field(p, f' TOC \\o "{cfg.TOC_LEVELS}" \\h \\z \\u ',
              "Right-click here and choose Update Field to build the contents.")


def kv_line(doc, key, value, mono=True):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(1)
    kr = p.add_run(f"{key}: "); kr.bold = True; kr.font.size = Pt(9)
    vr = p.add_run(str(value)); vr.font.size = Pt(9)
    if mono:
        vr.font.name = cfg.MONO_FONT


def page_break(doc):
    doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)

print("setup_document / add_toc / kv_line / page_break defined")


## 17. Front matter — title page, contents, index

The index table is the practical entry point: 50 rows, one per repository, with
a `Card` column that says at a glance whether anything is missing.


In [ ]:
doc = setup_document()

h = doc.add_heading(cfg.DOC_TITLE, level=0)
h.alignment = WD_ALIGN_PARAGRAPH.CENTER

sp = doc.add_paragraph(cfg.DOC_SUBTITLE)
sp.alignment = WD_ALIGN_PARAGRAPH.CENTER
sp.runs[0].italic = True

dp = doc.add_paragraph(f"Compiled {datetime.now():%d %B %Y, %H:%M} - "
                       f"{N_OK} of {len(ROWS)} model cards retrieved")
dp.alignment = WD_ALIGN_PARAGRAPH.CENTER
dp.runs[0].font.size = Pt(9)
add_hrule(doc)

doc.add_heading("Contents", level=1)
add_toc(doc)
page_break(doc)

doc.add_heading("Index of models", level=1)
idx = [["#", "Part", "Group", "Variant", "Repository", "Card"]]
for k, (part, group, variant, repo) in enumerate(ROWS, 1):
    idx.append([str(k), part.split("-")[0].strip(), group, variant, repo,
                "yes" if CARDS[repo] is not None else "MISSING"])
add_table(doc, idx)

if MISSING:
    doc.add_heading("Repositories with no retrievable card", level=2)
    for r in MISSING:
        kv_line(doc, r, STATUS[r], mono=False)

page_break(doc)
print(f"front matter written: {len(doc.paragraphs)} paragraphs, "
      f"{len(doc.tables)} table(s)")


## 18. Body — one titled section per model

Heading levels 1-4 are Part / Group / Variant / model. The model heading keeps
the built-in `Heading 4` style so it appears in the TOC, but its run formatting
is overridden so the repository id reads as an unmistakable title rather than a
small italic sub-heading.


In [ ]:
t0 = time.time()

for part, groups in MODELS:
    doc.add_heading(part, level=1)

    for group, variants in groups:
        doc.add_heading(group, level=2)

        for variant, repos in variants:
            doc.add_heading(variant, level=3)

            for repo in repos:
                hh = doc.add_heading("", level=4)
                hr_ = hh.add_run(f"MODEL - {repo}")
                hr_.bold, hr_.italic = True, False
                hr_.font.size = Pt(13.5)
                hr_.font.color.rgb = RGBColor(0x1F, 0x3B, 0x63)
                hh.paragraph_format.space_before = Pt(10)

                kv_line(doc, "Repository", repo)
                kv_line(doc, "URL", f"https://huggingface.co/{repo}")
                m = METAS.get(repo) or {}
                if m.get("lastModified"): kv_line(doc, "Last modified", m["lastModified"])
                if m.get("sha"):          kv_line(doc, "Commit", m["sha"])
                if m.get("n_files"):      kv_line(doc, "Files", f"{m['n_files']} - {m.get('files','')}")
                add_hrule(doc)

                body = CARDS[repo]
                if body is None:
                    p = doc.add_paragraph()
                    r = p.add_run(f"Model card unavailable - {STATUS[repo]}")
                    r.bold = True
                    r.font.color.rgb = RGBColor(0xC0, 0x00, 0x00)
                    if cfg.PAGE_BREAK_PER_MODEL:
                        page_break(doc)
                    continue

                fm = FRONTMATTER_RE.match(body)
                if fm and cfg.KEEP_FRONTMATTER:
                    doc.add_heading("YAML metadata (card frontmatter)", level=5)
                    add_code_block(doc, fm.group(1).split("\n"), "yaml")
                if fm:
                    body = body[fm.end():]

                doc.add_heading("Model card", level=5)
                render_markdown(doc, body, base_level=6)

                if cfg.PAGE_BREAK_PER_MODEL:
                    page_break(doc)

print(f"body written in {time.time() - t0:.1f} s")
print(f"total: {len(doc.paragraphs)} paragraphs, {len(doc.tables)} tables")


## 19. Save the document


In [ ]:
doc.save(DOCX_PATH)
size_mb = DOCX_PATH.stat().st_size / 1e6
print(f"saved {DOCX_PATH}  ({size_mb:.2f} MB)")


## 20. Flat markdown bundle

The same content as one `.md` file. For actually writing the Results section
this is the more useful artefact — it is plain text, so numbers can be read out
of it directly, whereas a `.docx` is a zip of XML.


In [ ]:
parts = [f"# {cfg.DOC_TITLE}", "", cfg.DOC_SUBTITLE, "",
         f"Generated {datetime.now():%Y-%m-%d %H:%M} - "
         f"{N_OK}/{len(ROWS)} cards retrieved", ""]

for part, groups in MODELS:
    parts += ["", "---", "", f"# {part}", ""]
    for group, variants in groups:
        parts += [f"## {group}", ""]
        for variant, repos in variants:
            parts += [f"### {variant}", ""]
            for repo in repos:
                parts += [f"#### MODEL: {repo}", "",
                          f"URL: https://huggingface.co/{repo}"]
                m = METAS.get(repo) or {}
                if m.get("lastModified"):
                    parts.append(f"Last modified: {m['lastModified']}  "
                                 f"Commit: {m.get('sha','')}")
                parts += [""]
                body = CARDS[repo]
                parts += [body if body else
                          f"*** CARD UNAVAILABLE - {STATUS[repo]} ***", ""]

MD_PATH.write_text("\n".join(parts), encoding="utf-8")
print(f"saved {MD_PATH}  ({MD_PATH.stat().st_size/1e6:.2f} MB, "
      f"{len(parts)} blocks)")


## 21. Verification

Re-open the saved file and check it against the registry. A document that opens
but silently lost half its models is worse than one that fails loudly.


In [ ]:
if cfg.VERIFY:
    chk = Document(DOCX_PATH)

    heads = [(p.style.name, p.text.strip()) for p in chk.paragraphs
             if p.style.name.startswith("Heading")]
    h4 = [t for s, t in heads if s == "Heading 4" and t.startswith("MODEL - ")]
    found = {t.replace("MODEL - ", "") for t in h4}

    missing_sections = set(REPOS) - found
    extra_sections = found - set(REPOS)

    print(f"paragraphs     : {len(chk.paragraphs)}")
    print(f"tables         : {len(chk.tables)}")
    print(f"headings       : {len(heads)}")
    print(f"model sections : {len(found)} / {len(REPOS)}")

    assert not missing_sections, f"no section for: {sorted(missing_sections)}"
    assert not extra_sections, f"unexpected section: {sorted(extra_sections)}"

    lvl1 = [t for s, t in heads if s == "Heading 1"]
    print(f"level-1        : {lvl1}")

    for repo in REPOS:
        if CARDS[repo]:
            assert repo in MD_PATH.read_text(encoding="utf-8"), \
                f"{repo} absent from the markdown bundle"

    print("\nVERIFY PASSED - every registry entry has its own titled section")
else:
    print("verification skipped (CFG.VERIFY False)")


## 22. Per-model content audit

Counts what actually landed under each model heading. A card that fetched fine
but rendered as three empty paragraphs would pass section 21 and still be
useless, so the content is measured here.


In [ ]:
audit = []
for part, group, variant, repo in ROWS:
    body = CARDS[repo] or ""
    n_tables = sum(1 for i, ln in enumerate(body.split("\n")[:-1])
                   if "|" in ln and TABLE_SEP_RE.match(body.split("\n")[i + 1]))
    audit.append({
        "repo": repo,
        "chars": len(body),
        "lines": body.count("\n"),
        "yaml": bool(FRONTMATTER_RE.match(body)),
        "tables": n_tables,
        "code": body.count("```") // 2,
        "status": STATUS[repo],
    })

w = max(len(a["repo"]) for a in audit)
print(f"{'repo':<{w}}  {'chars':>6} {'lines':>5} {'yaml':>5} "
      f"{'tbl':>4} {'code':>4}  status")
for a in audit:
    print(f"{a['repo']:<{w}}  {a['chars']:>6} {a['lines']:>5} "
          f"{str(a['yaml']):>5} {a['tables']:>4} {a['code']:>4}  {a['status']}")

thin = [a["repo"] for a in audit if 0 < a["chars"] < 400]
if thin:
    print(f"\nWARNING - {len(thin)} card(s) under 400 chars, probably stubs:")
    for r in thin:
        print(f"  {r}")

print(f"\ntotal cards with a results table: "
      f"{sum(1 for a in audit if a['tables'] > 0)}/{len(audit)}")


## 23. Run manifest

Small JSON recording exactly which commit of each repository this document was
built from. Worth keeping: cards get edited, and the Results section should be
traceable to a specific state of the Hub.


In [ ]:
manifest = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "n_repos": len(ROWS),
    "n_ok": N_OK,
    "n_missing": len(MISSING),
    "authenticated": bool(HF_TOKEN),
    "docx": cfg.DOCX_NAME,
    "md": cfg.MD_NAME,
    "cfg": {k: (list(v) if isinstance(v, tuple) else v)
            for k, v in asdict(cfg).items()},
    "models": [
        {"part": p, "group": g, "variant": v, "repo": r,
         "status": STATUS[r],
         "sha": (METAS.get(r) or {}).get("sha", ""),
         "lastModified": (METAS.get(r) or {}).get("lastModified", ""),
         "chars": len(CARDS[r] or "")}
        for p, g, v, r in ROWS
    ],
}

MANIFEST_PATH = OUT / "model_card_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"saved {MANIFEST_PATH}")
print(json.dumps({k: manifest[k] for k in
                  ("generated", "n_repos", "n_ok", "n_missing",
                   "authenticated")}, indent=2))


## 24. Outputs

Everything sits in `/kaggle/working/` and appears under the notebook's **Output**
tab after the session is saved (Save Version → download from there).


In [ ]:
print(f"{'file':<46} {'MB':>8}")
print("-" * 56)
total = 0
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        mb = p.stat().st_size / 1e6
        total += mb
        rel = str(p.relative_to(OUT))
        if rel.startswith(cfg.CACHE_DIRNAME):
            continue
        print(f"{rel:<46} {mb:>8.3f}")
n_cached = len(list(CACHE.glob('*.md')))
print(f"{cfg.CACHE_DIRNAME + f'/ ({n_cached} files)':<46} "
      f"{sum(p.stat().st_size for p in CACHE.glob('*.md'))/1e6:>8.3f}")
print("-" * 56)
print(f"{'total':<46} {total:>8.3f}")

try:
    from IPython.display import FileLink, display
    display(FileLink(str(DOCX_PATH)))
    display(FileLink(str(MD_PATH)))
except Exception:
    pass


## 25. Checklist and next step

**Verify before using the output**

- [ ] Section 10 reported **0 missing** cards. If the five `-R1-` Stage-4 repos
      came back 404, the ids in section 5 are wrong — fix them there and re-run.
      Without the R1 arm there is no R0-vs-R1 comparison to write up.
- [ ] Section 22 flagged no cards under 400 characters. A stub card means that
      run's numbers were never written to the Hub and will have to come from the
      `results_*.json` file instead.
- [ ] Section 21 printed `VERIFY PASSED`.
- [ ] Open the `.docx` in Word and hit **Update Field** on the Contents page —
      the TOC is a field and is empty until Word builds it.

**What is in the document**

Part → architecture/stage → split/arm → `MODEL - owner/repo`, and under each
model: repo id, URL, last-modified date, commit sha, file list, the YAML
frontmatter verbatim (this is where `model-index` metrics live), then the full
rendered card.

**Next step**

Hand over `GeoNUSAF_All_Model_Cards.md` rather than the `.docx` — identical
content, but plain text, so the per-class IoU / mIoU / clDice tables can be read
straight out of it for the Results section. Attach
`model_card_manifest.json` alongside it if you want the numbers tied to specific
commits.


In [ ]:
print("=" * 62)
print("GeoNUSAF model card compendium - run complete")
print("=" * 62)
print(f"repositories        : {len(ROWS)}")
print(f"cards retrieved     : {N_OK}")
print(f"cards missing       : {len(MISSING)}")
for r in MISSING:
    print(f"    MISSING  {r}  ({STATUS[r]})")
print(f"docx                : {DOCX_PATH.name} "
      f"({DOCX_PATH.stat().st_size/1e6:.2f} MB)")
print(f"markdown            : {MD_PATH.name} "
      f"({MD_PATH.stat().st_size/1e6:.2f} MB)")
print(f"manifest            : {MANIFEST_PATH.name}")
print(f"raw card cache      : {cfg.CACHE_DIRNAME}/ "
      f"({len(list(CACHE.glob('*.md')))} files)")
print("=" * 62)
